In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
rng = np.random.default_rng(42)
print("ready")

ready


In [3]:
n_customers = 20000

customers = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "age": rng.integers(20, 75, n_customers),
    "annual_income_eur": rng.lognormal(10.7, 0.4, n_customers).round(-2),
    "employment_type": rng.choice(
        ["permanent", "temporary", "self_employed", "unemployed"],
        size=n_customers, p=[0.65, 0.15, 0.12, 0.08]),
    "region": rng.choice(
        ["East", "South", "Central", "North", "West"],
        size=n_customers, p=[0.45, 0.12, 0.15, 0.08, 0.20]),
})

print(customers.shape)
customers.head()

(20000, 5)


,customer_id,age,annual_income_eur,employment_type,region
0,1,24,20900.0,permanent,East
1,2,62,69700.0,permanent,East
2,3,56,71200.0,permanent,East
3,4,44,36700.0,temporary,East
4,5,43,54600.0,self_employed,East


In [4]:
borrowers = customers.sample(frac=0.6, random_state=42).reset_index(drop=True)
n_loans = len(borrowers)

loans = pd.DataFrame({
    "loan_id": np.arange(1, n_loans + 1),
    "customer_id": borrowers["customer_id"],
    "loan_type": rng.choice(
        ["mortgage", "car", "consumer"],
        size=n_loans, p=[0.50, 0.25, 0.25]),
})

print(loans.shape)
loans.head()

(12000, 3)


,loan_id,customer_id,loan_type
0,1,10651,consumer
1,2,2042,mortgage
2,3,8669,consumer
3,4,1115,car
4,5,13903,mortgage


In [5]:
t = loans["loan_type"]

loans["amount_eur"] = np.select(
    [t == "mortgage", t == "car", t == "consumer"],
    [rng.uniform(100_000, 500_000, n_loans),
     rng.uniform(10_000, 50_000, n_loans),
     rng.uniform(2_000, 30_000, n_loans)],
).round(-2)

print(loans.shape)
loans.head()

(12000, 4)


,loan_id,customer_id,loan_type,amount_eur
0,1,10651,consumer,15200.0
1,2,2042,mortgage,316700.0
2,3,8669,consumer,27100.0
3,4,1115,car,48200.0
4,5,13903,mortgage,357700.0


In [6]:
t = loans["loan_type"]
income = borrowers["annual_income_eur"]

multiple = np.select(
    [t == "mortgage", t == "car", t == "consumer"],
    [rng.uniform(2.0, 5.0, n_loans),
     rng.uniform(0.2, 1.0, n_loans),
     rng.uniform(0.05, 0.5, n_loans)],
)

loans["amount_eur"] = (income * multiple).round(-2)

print(loans.shape)
loans.head()

(12000, 4)


,loan_id,customer_id,loan_type,amount_eur
0,1,10651,consumer,10500.0
1,2,2042,mortgage,593700.0
2,3,8669,consumer,4900.0
3,4,1115,car,21400.0
4,5,13903,mortgage,87800.0


In [7]:
ratio = loans["amount_eur"] / income
print(round(ratio[t == "mortgage"].max(), 2))
print(income[1])

5.0
123100.0


In [8]:
loans["interest_rate"] = np.select(
    [t == "mortgage", t == "car", t == "consumer"],
    [rng.uniform(0.04, 0.06, n_loans),
     rng.uniform(0.06, 0.09, n_loans),
     rng.uniform(0.10, 0.20, n_loans)],
).round(4)

loans["term_months"] = np.select(
    [t == "mortgage", t == "car", t == "consumer"],
    [rng.choice([240, 300, 360], n_loans),
     rng.choice([36, 60, 84], n_loans),
     rng.choice([12, 24, 36, 60], n_loans)],
)

print(loans.shape)
loans.head()

(12000, 6)


,loan_id,customer_id,loan_type,amount_eur,interest_rate,term_months
0,1,10651,consumer,10500.0,0.1854,60
1,2,2042,mortgage,593700.0,0.0494,360
2,3,8669,consumer,4900.0,0.1938,12
3,4,1115,car,21400.0,0.0691,84
4,5,13903,mortgage,87800.0,0.0539,240


In [9]:
r = loans["interest_rate"] / 12
n = loans["term_months"]

loans["monthly_payment_eur"] = (
    loans["amount_eur"] * r / (1 - (1 + r) ** -n)
).round(0)

pti = loans["monthly_payment_eur"] * 12 / income

print(loans["monthly_payment_eur"][1])
print(round(pti[1], 2))

3165.0
0.31


In [10]:
emp = borrowers["employment_type"]
age = borrowers["age"]

score = (
    -4.2
    + 4.0 * pti
    + np.select([emp == "unemployed", emp == "temporary", emp == "self_employed"],
                [1.5, 0.5, 0.4], default=0)
    + np.where(age < 25, 0.4, 0)
    + np.where(t == "consumer", 0.6, 0)
    + rng.normal(0, 0.5, n_loans)
)

p_default = 1 / (1 + np.exp(-score))
loans["defaulted"] = (rng.uniform(0, 1, n_loans) < p_default).astype(int)

print(round(loans["defaulted"].mean(), 3))
print(loans["defaulted"].sum())

0.061
732


In [11]:
n_tx = rng.poisson(40, n_customers)
tx_customer = np.repeat(customers["customer_id"].values, n_tx)
n_total = len(tx_customer)

start = pd.Timestamp("2026-04-01")
seconds = rng.integers(0, 183 * 24 * 3600, n_total)

transactions = pd.DataFrame({
    "transaction_id": np.arange(1, n_total + 1),
    "customer_id": tx_customer,
    "timestamp": start + pd.to_timedelta(seconds, unit="s"),
    "transaction_type": rng.choice(
        ["card_payment", "transfer_out", "transfer_in", "cash_deposit", "cash_withdrawal"],
        size=n_total, p=[0.60, 0.15, 0.15, 0.05, 0.05]),
})

print(transactions.shape)
transactions.head()

(800988, 4)


,transaction_id,customer_id,timestamp,transaction_type
0,1,1,2026-04-11 12:16:12,transfer_out
1,2,1,2026-07-17 00:39:59,transfer_out
2,3,1,2026-04-21 07:22:56,card_payment
3,4,1,2026-07-14 11:15:26,card_payment
4,5,1,2026-06-30 06:10:35,card_payment


In [12]:
tt = transactions["transaction_type"]

transactions["amount_eur"] = np.select(
    [tt == "card_payment", tt == "cash_withdrawal", tt == "cash_deposit",
     tt == "transfer_out", tt == "transfer_in"],
    [rng.lognormal(3.4, 1.0, n_total),
     rng.lognormal(4.5, 0.7, n_total),
     rng.lognormal(5.0, 1.0, n_total),
     rng.lognormal(5.5, 1.0, n_total),
     rng.lognormal(6.5, 0.8, n_total)],
).round(2)

transactions = transactions.sort_values(["customer_id", "timestamp"]).reset_index(drop=True)
transactions["transaction_id"] = np.arange(1, n_total + 1)

print(transactions.shape)
transactions.head()

(800988, 5)


,transaction_id,customer_id,timestamp,transaction_type,amount_eur
0,1,1,2026-04-09 20:20:24,card_payment,14.36
1,2,1,2026-04-11 12:16:12,transfer_out,576.90
2,3,1,2026-04-21 07:22:56,card_payment,10.98
3,4,1,2026-04-22 17:05:01,transfer_in,1226.13
4,5,1,2026-05-05 10:25:09,card_payment,29.13


In [13]:
structurers = rng.choice(customers["customer_id"], 40, replace=False)

rows = []
for cid in structurers:
    k = rng.integers(6, 13)
    day0 = start + pd.Timedelta(days=int(rng.integers(0, 165)))
    for s in rng.integers(0, 14 * 24 * 3600, k):
        rows.append({
            "customer_id": cid,
            "timestamp": day0 + pd.Timedelta(seconds=int(s)),
            "transaction_type": "cash_deposit",
            "amount_eur": round(rng.uniform(9000, 9990), 2),
        })

structuring = pd.DataFrame(rows)
print(structuring.shape)
structuring.head()

(370, 4)


,customer_id,timestamp,transaction_type,amount_eur
0,18127,2026-04-09 23:57:02,cash_deposit,9163.92
1,18127,2026-04-10 19:58:07,cash_deposit,9030.67
2,18127,2026-04-11 18:26:54,cash_deposit,9443.85
3,18127,2026-04-14 20:43:37,cash_deposit,9154.07
4,18127,2026-04-15 21:14:09,cash_deposit,9353.32


In [14]:
mules = rng.choice(customers["customer_id"], 30, replace=False)

rows = []
for cid in mules:
    for _ in range(rng.integers(2, 5)):
        t_in = start + pd.Timedelta(seconds=int(rng.integers(0, 180 * 24 * 3600)))
        amount_in = round(rng.uniform(15000, 50000), 2)
        rows.append({"customer_id": cid, "timestamp": t_in,
                     "transaction_type": "transfer_in", "amount_eur": amount_in})
        rows.append({"customer_id": cid,
                     "timestamp": t_in + pd.Timedelta(hours=int(rng.integers(1, 24))),
                     "transaction_type": "transfer_out",
                     "amount_eur": round(amount_in * rng.uniform(0.90, 0.99), 2)})

pass_through = pd.DataFrame(rows)
print(pass_through.shape)
pass_through.head()

(170, 4)


,customer_id,timestamp,transaction_type,amount_eur
0,7989,2026-09-07 00:59:40,transfer_in,22839.52
1,7989,2026-09-07 19:59:40,transfer_out,22482.86
2,7989,2026-08-12 09:29:58,transfer_in,28274.58
3,7989,2026-08-13 08:29:58,transfer_out,27102.40
4,7989,2026-06-26 14:40:06,transfer_in,15561.10


In [15]:
aml_truth = pd.concat([
    pd.DataFrame({"customer_id": structurers, "pattern": "structuring"}),
    pd.DataFrame({"customer_id": mules, "pattern": "pass_through"}),
], ignore_index=True)

transactions = pd.concat(
    [transactions.drop(columns="transaction_id"), structuring, pass_through],
    ignore_index=True,
)
transactions = transactions.sort_values(["customer_id", "timestamp"]).reset_index(drop=True)
transactions.insert(0, "transaction_id", np.arange(1, len(transactions) + 1))

print(transactions.shape)
print(aml_truth.shape)

(801528, 5)
(70, 2)


In [16]:
os.makedirs("data", exist_ok=True)
os.makedirs("answer_key", exist_ok=True)

customers.to_csv("data/customers.csv", index=False)
loans.to_csv("data/loans.csv", index=False)
transactions.to_csv("data/transactions.csv", index=False)
aml_truth.to_csv("answer_key/aml_truth.csv", index=False)

for f in os.listdir("data"):
    size = os.path.getsize(f"data/{f}") / 1_000_000
    print(f, round(size, 1), "MB")

customers.csv 0.7 MB
loans.csv 0.6 MB
sample 0.0 MB
transactions.csv 42.1 MB


In [17]:
os.makedirs("data/sample", exist_ok=True)

customers.head(1000).to_csv("data/sample/customers_sample.csv", index=False)
loans.head(1000).to_csv("data/sample/loans_sample.csv", index=False)
transactions.head(1000).to_csv("data/sample/transactions_sample.csv", index=False)

print(os.listdir("data/sample"))

['customers_sample.csv', 'loans_sample.csv', 'transactions_sample.csv']
